# Part 3 — Churn Prediction Model & Model Card
This notebook is generated from `train_churn_model.py` and documents the actual run outputs.

## Data Loading

The workflow loads `rfm_modeling_snapshot.csv` from a relative data directory and uses the actual run output below.

| metric             | value                     |
|:-------------------|:--------------------------|
| source_table       | rfm_modeling_snapshot.csv |
| rows               | 2400                      |
| columns            | 29                        |
| snapshot_date_min  | 2025-09-30                |
| snapshot_date_max  | 2025-09-30                |
| overall_churn_rate | 0.4696                    |

In [ ]:
from pathlib import Path
import pandas as pd

candidates = [
    Path('data'),
    Path('../d2c churn data package/d2c churn data package'),
]
data_dir = next(path for path in candidates if (path / 'rfm_modeling_snapshot.csv').exists())
df = pd.read_csv(data_dir / 'rfm_modeling_snapshot.csv')
df.head()

## Feature Preparation

The target and non-feature identifiers are excluded before model training.

| metric               | value                                             |
|:---------------------|:--------------------------------------------------|
| total_model_features | 25                                                |
| numeric_features     | 19                                                |
| categorical_features | 6                                                 |
| excluded_columns     | customer_id, snapshot_date, churn_next_60d, split |

## Leakage Checks

The following columns were excluded because they would leak outcome, split assignment, or row identity into the model.

| column         | reason_excluded                                                                   |
|:---------------|:----------------------------------------------------------------------------------|
| customer_id    | row identifier; would let the model memorize customers rather than learn behavior |
| snapshot_date  | snapshot metadata, not customer behavior; constant-time anchor for this run       |
| split          | evaluation assignment; not available at scoring time                              |
| churn_next_60d | future outcome target and direct leakage                                          |

In [ ]:
feature_cols = [
    c for c in df.columns
    if c not in ['customer_id', 'snapshot_date', 'churn_next_60d', 'split']
]
categorical_cols = df[feature_cols].select_dtypes(include=['object']).columns.tolist()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]
feature_cols[:5], len(numeric_cols), len(categorical_cols)

## Train / Validation / Test Split

The provided split column is preserved exactly.

| split      |   customers |   churn_rate |
|:-----------|------------:|-------------:|
| test       |         336 |       0.5    |
| train      |        1728 |       0.4699 |
| validation |         336 |       0.4375 |

In [ ]:
train = df.loc[df['split'] == 'train'].copy()
validation = df.loc[df['split'] == 'validation'].copy()
test = df.loc[df['split'] == 'test'].copy()
train.shape, validation.shape, test.shape

## Candidate Models

The notebook compares a simple baseline model against a stronger tree-based challenger.

1. **Baseline model:** logistic regression with scaled numeric features and one-hot encoded categoricals.
2. **Stronger model:** histogram gradient boosting with ordinal-encoded categoricals.

## Model Comparison

| model_name             | model_family        |   threshold |   accuracy |   roc_auc |   pr_auc |   precision |   recall |     f1 |   positive_rate |   tn |   fp |   fn |   tp |
|:-----------------------|:--------------------|------------:|-----------:|----------:|---------:|------------:|---------:|-------:|----------------:|-----:|-----:|-----:|-----:|
| logistic_regression    | baseline            |         0.5 |     0.8155 |    0.8827 |   0.8676 |      0.8058 |   0.7619 | 0.7832 |          0.4137 |  162 |   27 |   35 |  112 |
| hist_gradient_boosting | stronger challenger |         0.5 |     0.7976 |    0.876  |   0.8573 |      0.7687 |   0.7687 | 0.7687 |          0.4375 |  155 |   34 |   34 |  113 |

In [ ]:
# Baseline: LogisticRegression
# Stronger challenger: HistGradientBoostingClassifier
# Both models are trained in train_churn_model.py and compared on validation PR-AUC.

## Evaluation Metrics

Validation operating-point metrics:

| metric        |    value |
|:--------------|---------:|
| accuracy      |   0.7946 |
| roc_auc       |   0.8827 |
| pr_auc        |   0.8676 |
| precision     |   0.89   |
| recall        |   0.6054 |
| f1            |   0.7206 |
| positive_rate |   0.2976 |
| tn            | 178      |
| fp            |  11      |
| fn            |  58      |
| tp            |  89      |

Test operating-point metrics:

| metric        |    value |
|:--------------|---------:|
| accuracy      |   0.7649 |
| roc_auc       |   0.8845 |
| pr_auc        |   0.8778 |
| precision     |   0.8803 |
| recall        |   0.6131 |
| f1            |   0.7228 |
| positive_rate |   0.3482 |
| tn            | 154      |
| fp            |  14      |
| fn            |  65      |
| tp            | 103      |

## Threshold Selection

The final model is **logistic_regression** and the selected threshold is **0.705**. Thresholds were swept on the validation set, and the final operating point had to satisfy the CRM capacity rule of positive rate <= 30%.

Top candidate thresholds by validation F1:

|   threshold |   accuracy |   roc_auc |   pr_auc |   precision |   recall |     f1 |   positive_rate |   tn |   fp |   fn |   tp |
|------------:|-----------:|----------:|---------:|------------:|---------:|-------:|----------------:|-----:|-----:|-----:|-----:|
|       0.4   |     0.8065 |    0.8827 |   0.8676 |      0.75   |   0.8367 | 0.791  |          0.4881 |  148 |   41 |   24 |  123 |
|       0.45  |     0.8155 |    0.8827 |   0.8676 |      0.7891 |   0.7891 | 0.7891 |          0.4375 |  158 |   31 |   31 |  116 |
|       0.39  |     0.8036 |    0.8827 |   0.8676 |      0.7455 |   0.8367 | 0.7885 |          0.4911 |  147 |   42 |   24 |  123 |
|       0.395 |     0.8036 |    0.8827 |   0.8676 |      0.7455 |   0.8367 | 0.7885 |          0.4911 |  147 |   42 |   24 |  123 |
|       0.43  |     0.8125 |    0.8827 |   0.8676 |      0.78   |   0.7959 | 0.7879 |          0.4464 |  156 |   33 |   30 |  117 |
|       0.485 |     0.8185 |    0.8827 |   0.8676 |      0.8071 |   0.7687 | 0.7875 |          0.4167 |  162 |   27 |   34 |  113 |
|       0.445 |     0.8125 |    0.8827 |   0.8676 |      0.7838 |   0.7891 | 0.7864 |          0.4405 |  157 |   32 |   31 |  116 |
|       0.515 |     0.8185 |    0.8827 |   0.8676 |      0.8116 |   0.7619 | 0.786  |          0.4107 |  163 |   26 |   35 |  112 |
|       0.51  |     0.8185 |    0.8827 |   0.8676 |      0.8116 |   0.7619 | 0.786  |          0.4107 |  163 |   26 |   35 |  112 |
|       0.385 |     0.8006 |    0.8827 |   0.8676 |      0.741  |   0.8367 | 0.7859 |          0.494  |  146 |   43 |   24 |  123 |

## Visual Diagnostics

### Validation ROC Curve
![Validation ROC](charts/01_validation_roc_curve.png)

### Validation Precision-Recall Curve
![Validation PR](charts/02_validation_pr_curve.png)

### Validation Confusion Matrix
![Validation Confusion](charts/03_validation_confusion_matrix.png)

### Top Model Drivers
![Feature Importance](charts/04_top_model_drivers.png)

## Final Model Saving

The selected pipeline is serialized to `model.pkl` together with the threshold and feature column list.

| artifact_field   | value               |
|:-----------------|:--------------------|
| path             | model.pkl           |
| model_name       | logistic_regression |
| threshold        | 0.705               |
| feature_count    | 25                  |

In [ ]:
import joblib

loaded = joblib.load('model.pkl')
loaded.keys(), loaded['model_name'], loaded['threshold']

# Error Analysis

The table below lists five false positives and five false negatives from the test split at the selected business threshold.

| customer_id   | error_type     |   predicted_probability |   churn_next_60d |   recency_days |   frequency_180d |   monetary_180d |   ticket_count_90d |   sessions_30d |   last_visit_days_ago | case_note                                                                                                                                        | business_risk                                                                               |
|:--------------|:---------------|------------------------:|-----------------:|---------------:|-----------------:|----------------:|-------------------:|---------------:|----------------------:|:-------------------------------------------------------------------------------------------------------------------------------------------------|:--------------------------------------------------------------------------------------------|
| CUST01246     | False Positive |                  0.9821 |                0 |            262 |                0 |            0    |                  0 |              1 |                    60 | Model flagged churn because of very stale last order, weak recent activity, only one order in the last 180 days.                                 | Unnecessary retention spend or outreach fatigue on a customer who would have stayed anyway. |
| CUST01325     | False Positive |                  0.9563 |                0 |            186 |                0 |            0    |                  0 |              1 |                    43 | Model flagged churn because of very stale last order, weak recent activity, only one order in the last 180 days.                                 | Unnecessary retention spend or outreach fatigue on a customer who would have stayed anyway. |
| CUST01411     | False Positive |                  0.9372 |                0 |            183 |                0 |            0    |                  0 |              0 |                    51 | Model flagged churn because of very stale last order, weak recent activity, only one order in the last 180 days.                                 | Unnecessary retention spend or outreach fatigue on a customer who would have stayed anyway. |
| CUST00437     | False Positive |                  0.9328 |                0 |            151 |                1 |          729.22 |                  0 |              0 |                    33 | Model flagged churn because of very stale last order, weak recent activity, only one order in the last 180 days.                                 | Unnecessary retention spend or outreach fatigue on a customer who would have stayed anyway. |
| CUST01370     | False Positive |                  0.8893 |                0 |            161 |                2 |         1246.04 |                  0 |              2 |                    35 | Model flagged churn because of very stale last order, weak recent activity, no campaign response.                                                | Unnecessary retention spend or outreach fatigue on a customer who would have stayed anyway. |
| CUST02072     | False Negative |                  0.0475 |                1 |             35 |                7 |         4340.19 |                  0 |              4 |                     1 | Model missed churn because stronger positive signals masked no campaign response.                                                                | Missed intervention on a real churner, creating avoidable revenue loss and lower save-rate. |
| CUST01990     | False Negative |                  0.0837 |                1 |             59 |                4 |         3877.77 |                  0 |             11 |                     7 | Model missed churn because stronger positive signals masked mixed signals.                                                                       | Missed intervention on a real churner, creating avoidable revenue loss and lower save-rate. |
| CUST01028     | False Negative |                  0.6994 |                1 |            173 |                1 |         2809.6  |                  0 |              6 |                    45 | Model missed churn because stronger positive signals masked very stale last order, only one order in the last 180 days, no campaign response.    | Missed intervention on a real churner, creating avoidable revenue loss and lower save-rate. |
| CUST00507     | False Negative |                  0.6882 |                1 |            120 |                1 |         2716.4  |                  0 |              1 |                    26 | Model missed churn because stronger positive signals masked recency already slipping, weak recent activity, only one order in the last 180 days. | Missed intervention on a real churner, creating avoidable revenue loss and lower save-rate. |
| CUST00438     | False Negative |                  0.564  |                1 |             64 |                3 |         2466.39 |                  2 |              6 |                    22 | Model missed churn because stronger positive signals masked recency already slipping, elevated return rate, negative support friction.           | Missed intervention on a real churner, creating avoidable revenue loss and lower save-rate. |

## Interpretation

1. **False positives** are usually customers whose recency and engagement looked weak, but who still came back in the target window.
2. **False negatives** tend to be customers with one or two still-positive signals, like moderate recency or engagement, that were not enough to offset their eventual churn.
3. The remaining opportunity is less about class balance and more about modeling contradictory signals such as recent browsing plus worsening order cadence.

## Business Risk By Error Type

1. **False positive risk:** the team may waste discount budget, contact capacity, or goodwill on customers who would have stayed without intervention.
2. **False negative risk:** the team misses a real save opportunity, which can directly reduce retained revenue and hide emerging churn patterns.


# Model Card

## Model Details

- **Model name:** logistic_regression
- **Model type:** Logistic regression baseline selected as final model because it outperformed the stronger tree-based challenger on validation PR-AUC.
- **Snapshot date:** 2025-09-30
- **Target:** `churn_next_60d`
- **Decision threshold:** 0.70

## Intended Use

This model is designed for internal CRM prioritization. It should rank customers for retention review and campaign routing, not automate customer-facing decisions without human oversight.

## Model Approach

- Input table: `rfm_modeling_snapshot.csv` with behavioral, support, returns, campaign, and profile aggregates at the 2025-09-30 snapshot
- Candidate models: logistic-regression baseline and a `HistGradientBoostingClassifier` challenger
- Preprocessing: median imputation for numeric features, categorical imputation for missing labels, one-hot encoding for logistic regression, and ordinal encoding for the tree-based challenger
- Selection rule: choose the model with the best validation PR-AUC, then set the operating threshold on the validation set under the CRM capacity constraint

## Data

- Source table: `rfm_modeling_snapshot.csv`
- Universe: 2,400 customers
- Split strategy: provided `train` / `validation` / `test`
- Leakage control: `churn_next_60d`, `split`, `customer_id`, and `snapshot_date` were excluded from features

## Performance

### Validation Comparison

| model_name             | model_family        |   threshold |   accuracy |   roc_auc |   pr_auc |   precision |   recall |     f1 |   positive_rate |   tn |   fp |   fn |   tp |
|:-----------------------|:--------------------|------------:|-----------:|----------:|---------:|------------:|---------:|-------:|----------------:|-----:|-----:|-----:|-----:|
| logistic_regression    | baseline            |         0.5 |     0.8155 |    0.8827 |   0.8676 |      0.8058 |   0.7619 | 0.7832 |          0.4137 |  162 |   27 |   35 |  112 |
| hist_gradient_boosting | stronger challenger |         0.5 |     0.7976 |    0.876  |   0.8573 |      0.7687 |   0.7687 | 0.7687 |          0.4375 |  155 |   34 |   34 |  113 |

### Final Operating Point

- Validation precision: 0.8900
- Validation recall: 0.6054
- Validation F1: 0.7206
- Validation accuracy: 0.7946
- Validation positive rate: 0.2976
- Test precision: 0.8803
- Test recall: 0.6131
- Test F1: 0.7228
- Test accuracy: 0.7649
- Test ROC-AUC: 0.8845
- Test PR-AUC: 0.8778

## Key Drivers

Positive coefficients raise the churn score:

| feature                       |   coefficient |
|:------------------------------|--------------:|
| num__recency_days             |        1.7222 |
| num__return_rate_180d         |        0.344  |
| num__negative_ticket_rate_90d |        0.301  |
| num__avg_discount_pct_180d    |        0.2939 |
| num__last_visit_days_ago      |        0.2912 |

Negative coefficients lower the churn score:

| feature                           |   coefficient |
|:----------------------------------|--------------:|
| num__monetary_180d                |       -0.4321 |
| cat__preferred_category_Fragrance |       -0.3864 |
| cat__acquisition_channel_Organic  |       -0.3841 |
| num__ticket_count_90d             |       -0.3065 |
| cat__loyalty_tier_Platinum        |       -0.2839 |

These are conditional model effects, not causal instructions. For example, `ticket_count_90d` becomes protective after controlling for recency and spend because some highly engaged customers also contact support.

## Limitations

1. The model is trained on one snapshot and may not generalize if campaign mix, pricing, or seasonality changes.
2. It sees behavioral aggregates, not raw customer intent; sudden life-cycle shifts can still be missed.
3. The score should not be interpreted as causal evidence that a discount or intervention will work.

## Ethical and Operational Risks

1. Marketing-heavy interventions may over-target paid-acquisition cohorts if scores are used without fairness review.
2. High-risk predictions can reflect service issues, not only customer disengagement, so the response should not default to discounts.
3. The model should not be used to deny service, downgrade support quality, or suppress loyal customers from legitimate help.

## When Not To Use This Model

1. Do not use it for punitive or adverse decisions such as denying support, degrading service, or removing customer benefits.
2. Do not use it as proof that a discount, outreach, or support action will cause retention; the score is predictive, not causal.
3. Do not use it when the feature snapshot is stale, upstream definitions have changed, or campaign policy has materially shifted without retraining.
4. Do not use it without human review for cases where support context or high customer value makes the intervention decision sensitive.

## Monitoring Needs

Track feature drift, prediction-rate drift, segment-wise precision, and the realized incremental retention lift from interventions triggered by the score.


In [ ]:
# Rebuild from the command line with: python train_churn_model.py